In [1]:
import sys
sys.path.append("..")
from src.fnn import FNN
from src.utils import device, polyak_update
from src.replay_buffer import ReplayBuffer
import torch
from torch import optim
from torch.nn.functional import mse_loss
from torch.nn.utils import clip_grad_norm_
import numpy as np
from tqdm import trange

# Test 1

In [ ]:
state_dim = 16
action_dim = 1
message_dim = 256
hidden_dim = 256
num_hidden_layers = 5
lr = 3e-4

message_actor_1 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_message_actor_1 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

message_actor_2 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_message_actor_2 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

final_actor_1 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_final_actor_1 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

final_actor_2 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_final_actor_2 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

critic_1 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_critic_1 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

critic_2 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_critic_2 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

online_models = [message_actor_1, message_actor_2, final_actor_1, final_actor_2, critic_1, critic_2]
target_models = [target_message_actor_1, target_message_actor_2, target_final_actor_1, target_final_actor_2, target_critic_1, target_critic_2]
optimizers = {}
for online, target in zip(online_models, target_models):
    optimizers[online] = optim.Adam(online.parameters(), lr = lr)
    target.load_state_dict(online.state_dict())
    for param in target.parameters():
        param.requires_grad_(False)

In [23]:
actor_tau = 0.005
critic_tau = 0.005
rho = 0.8
num_steps = 100000
update_every = 10
batch_size = 128
replay_capacity = 10000
epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay = 0.9999

In [47]:
def reward_fn(state, action):
    return -(state.sum(axis = -1, keepdims = True) - action) ** 2

In [49]:
torch.manual_seed(0)
rng = np.random.default_rng(0)
num_updates = 0             
epsilon = epsilon_start
rewards = []

replay_buffer_1= ReplayBuffer(
    capacity = replay_capacity,
    batch_size = batch_size,
    device = device,
    rng = rng,
    unsqueezed = False
)
replay_buffer_2 = ReplayBuffer(
    capacity = replay_capacity,
    batch_size = batch_size,
    device = device,
    rng = rng,
    unsqueezed = False
)
agent_1 = [
    message_actor_1,
    final_actor_1,
    critic_1,
    target_message_actor_1,
    target_final_actor_1,
    target_critic_1,
    replay_buffer_1,
]
agent_2 = [
    message_actor_2,
    final_actor_2,
    critic_2,
    target_message_actor_2,
    target_final_actor_2,
    target_critic_2,
    replay_buffer_2,
]

for i in range(num_steps):
    with torch.no_grad():
        state = torch.randn(state_dim).to(device)
        message = torch.zeros(message_dim).to(device)
        message[: state_dim] = state
        prev_message = torch.zeros_like(message)
        current_node = 1
        action = None
        lifetime = 0
        while True:
            lifetime += 1
            agent = agent_1 if current_node == 1 else agent_2
            next_node = 2 if current_node == 1 else 1
            *_, message_actor, final_actor, _, replay_buffer = agent
            if rng.random() < rho:
                next_message = message_actor(message)
                replay_buffer.add((state, prev_message, message, next_message))
                current_node = next_node
                prev_message = message
                message = next_message
            else:
                action = final_actor(message)
                reward = reward_fn(state, action)
                rewards.append(reward)
                break
    if i % update_every == 0:
        for agent, other_agent in [[agent_1, agent_2], [agent_2, agent_1]]:
            (
                message_actor,
                final_actor,
                critic,
                target_message_actor,
                target_final_actor,
                target_critic,
                replay_buffer,
            ) = agent
            *_, other_message_actor, other_final_actor, other_critic, _ = other_agent
            state, prev_message, message, next_message = replay_buffer.sample()

            with torch.no_grad():
                action = target_final_actor(message)
                reward = reward_fn(state, action)
                next_action = other_final_actor(next_message)
                next_reward = reward_fn(state, next_action)
                next_next_message = other_message_actor(next_message)
                other_critic_in = torch.cat([message, next_message, next_next_message], dim = 1)
                next_message_q = other_critic(other_critic_in)
                next_q = (1 - rho) * next_reward + rho * next_message_q
                target_q = (1 - rho) * reward + rho * next_q
            critic_in = torch.cat([prev_message, message, next_message], dim = 1)
            q = critic(critic_in)
            critic_loss = mse_loss(q, target_q)
            optimizers[critic].zero_grad()
            critic_loss.backward()
            clip_grad_norm_(critic.parameters(), 1.0)
            optimizers[critic].step()
            polyak_update(target_critic, critic, critic_tau)

            new_message = message_actor(message)
            critic_in = torch.cat([prev_message, message, new_message], dim = 1)
            q = critic(critic_in)
            message_actor_loss = -q.mean()
            optimizers[message_actor].zero_grad()
            message_actor_loss.backward()
            clip_grad_norm_(message_actor.parameters(), 1.0)
            optimizers[message_actor].step()
            polyak_update(target_message_actor, message_actor, actor_tau)

            action = final_actor(message)
            reward = reward_fn(state, action)
            final_actor_loss = -reward.mean()
            optimizers[final_actor].zero_grad()
            final_actor_loss.backward()
            clip_grad_norm_(final_actor.parameters(), 1.0)
            optimizers[final_actor].step()
            polyak_update(target_final_actor, final_actor, actor_tau)

            num_updates += 1
            if num_updates % 100 == 0:
                print(f"{i}, R: {reward.mean()}, L: {lifetime}")

KeyboardInterrupt: 

In [57]:
def test_connected_mutualism():
    state_dim = 16
    message_dim = 256
    action_dim = 1
    hidden_dim = 256
    num_hidden_layers = 5
    lr = 3e-4
    replay_capacity = 10000
    batch_size = 64

    message_actor = FNN(
        input_size=state_dim,
        output_size=message_dim,
        hidden_size=hidden_dim,
        num_hidden_layers=num_hidden_layers,
    ).to(device)
    target_message_actor = FNN(
        input_size=state_dim,
        output_size=message_dim,
        hidden_size=hidden_dim,
        num_hidden_layers=num_hidden_layers,
    ).to(device)
    target_message_actor.load_state_dict(message_actor.state_dict())
    for p in target_message_actor.parameters():
        p.requires_grad_(False)

    final_actor = FNN(
        input_size=message_dim,
        output_size=action_dim,
        hidden_size=hidden_dim,
        num_hidden_layers=num_hidden_layers,
    ).to(device)
    target_final_actor = FNN(
        input_size=message_dim,
        output_size=action_dim,
        hidden_size=hidden_dim,
        num_hidden_layers=num_hidden_layers,
    ).to(device)
    target_final_actor.load_state_dict(final_actor.state_dict())
    for p in target_final_actor.parameters():
        p.requires_grad_(False)

    message_critic = FNN(
        input_size=state_dim + message_dim,
        output_size=1,
        hidden_size=hidden_dim,
        num_hidden_layers=num_hidden_layers,
    ).to(device)
    target_message_critic = FNN(
        input_size=state_dim + message_dim,
        output_size=1,
        hidden_size=hidden_dim,
        num_hidden_layers=num_hidden_layers,
    ).to(device)
    target_message_critic.load_state_dict(message_critic.state_dict())
    for p in target_message_critic.parameters():
        p.requires_grad_(False)

    rng = np.random.default_rng(0)
    replay_buffer = ReplayBuffer(
        capacity=replay_capacity,
        batch_size=batch_size,
        device=device,
        rng=rng,
        unsqueezed=False,
    )

    opt_msg_actor = optim.Adam(message_actor.parameters(), lr=lr)
    opt_final_actor = optim.Adam(final_actor.parameters(), lr=lr)
    opt_critic = optim.Adam(message_critic.parameters(), lr=lr)

    for step in trange(20000, desc="connected mutualism"):
        state = torch.rand(batch_size, state_dim, device=device)

        # ===============================
        # Collect (state, msg) into replay buffer
        # ===============================
        with torch.no_grad():
            msg = target_message_actor(state)
        replay_buffer.add((state, msg))

        if replay_buffer.ready():
            # ===============================
            # 1) Train message critic from replay
            # ===============================
            state_b, msg_b = replay_buffer.sample()
            msg_b.requires_grad = True
            action_b = target_final_actor(msg_b)
            target_q = reward_fn(state_b, action_b)
            q = message_critic(torch.cat([state_b, msg_b], dim=-1))
            critic_loss = mse_loss(q, target_q)

            opt_critic.zero_grad(set_to_none=True)
            critic_loss.backward()
            opt_critic.step()

        # ===============================
        # 2) Train final actor via reward_fn directly
        # ===============================
        with torch.no_grad():
            msg = target_message_actor(state)
        action = final_actor(msg)
        final_actor_loss = -reward_fn(state, action).mean()

        opt_final_actor.zero_grad(set_to_none=True)
        final_actor_loss.backward()
        opt_final_actor.step()

        # ===============================
        # 3) Train message actor via message critic
        # ===============================
        msg = message_actor(state)
        q = target_message_critic(torch.cat([state, msg], dim=-1))
        msg_actor_loss = -q.mean()

        opt_msg_actor.zero_grad(set_to_none=True)
        msg_actor_loss.backward()
        opt_msg_actor.step()

        # ===============================
        # Polyak updates
        # ===============================
        polyak_update(target_message_actor, message_actor, polyak_factor=0.005)
        polyak_update(target_final_actor, final_actor, polyak_factor=0.005)
        polyak_update(target_message_critic, message_critic, polyak_factor=0.005)

        if step % 100 == 0:
            with torch.no_grad():
                msg = target_message_actor(state)
                action = target_final_actor(msg)
                r = reward_fn(state, action).mean()
            print(f"step {step} | reward: {r.item():.4f}")

test_connected_mutualism()

connected mutualism:   0%|          | 0/20000 [00:00<?, ?it/s]

connected mutualism:   0%|          | 7/20000 [00:00<09:37, 34.62it/s]

step 0 | reward: -63.6535


connected mutualism:   1%|          | 106/20000 [00:03<11:40, 28.39it/s]

step 100 | reward: -16.2611


...

connected mutualism:  99%|█████████▉| 19807/20000 [07:31<00:03, 50.48it/s]

step 19800 | reward: -0.4329


connected mutualism: 100%|█████████▉| 19908/20000 [07:33<00:01, 50.06it/s]

step 19900 | reward: -0.8068


connected mutualism: 100%|██████████| 20000/20000 [07:35<00:00, 43.89it/s]


# Test 2

In [2]:
state_dim = 16
action_dim = 1
message_dim = 256
hidden_dim = 256
num_hidden_layers = 5
lr = 3e-4

message_actor_1 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_message_actor_1 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

message_actor_2 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_message_actor_2 = FNN(
    input_size = message_dim,
    output_size = message_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

final_actor_1 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_final_actor_1 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

final_actor_2 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_final_actor_2 = FNN(
    input_size = message_dim,
    output_size = action_dim,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

critic_1 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_critic_1 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

critic_2 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)
target_critic_2 = FNN(
    input_size = 3 * message_dim,
    output_size = 1,
    hidden_size = hidden_dim,
    num_hidden_layers = num_hidden_layers,
).to(device)

online_models = [message_actor_1, message_actor_2, final_actor_1, final_actor_2, critic_1, critic_2]
target_models = [target_message_actor_1, target_message_actor_2, target_final_actor_1, target_final_actor_2, target_critic_1, target_critic_2]
optimizers = {}
for online, target in zip(online_models, target_models):
    optimizers[online] = optim.Adam(online.parameters(), lr = lr)
    target.load_state_dict(online.state_dict())
    for param in target.parameters():
        param.requires_grad_(False)